# Multiplexed Imaging Cell Phenotyping

Evaluate cell phenotyping on the **TNBC-MIBI dataset** [Keren et al., 2018]. The evaluation consists of three phases:

1. **Segmentation & AP Assessment** — generating composite images, running model inference, computing AP@0.5/0.75/0.9
2. **Expression Extraction & Comparison** — extracting per-cell mean intensity, arcsinh transforming, matching predicted to GT cells via IoU>0.5
3. **FlowSOM Hierarchical Clustering** — three-level immune phenotyping evaluated via Hungarian-matched accuracy

## Data Download

Download from [https://www.angelolab.com/mibi/data](https://www.angelolab.com/mibi/data) and place under `phenotyping/`:

| Data | Path | Description |
|---|---|---|
| Raw marker TIFs | `TNBC/TNBCShareData/Point{pid}/*.tif` | 40 protein markers, 1024x1024 float32 |
| GT segmentation masks | `TNBC_shareCellData/p{pid}_labeledcellData.tiff` | 41 uint16 label maps |
| Single-cell expression matrix | `TNBC_shareCellData/cellData.csv` | arcsinh-transformed, with cell type labels |
| Patient classification | `TNBC_shareCellData/patient_class.csv` | Molecular subtype |

In [ ]:
import sys
import time
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path

# Setup paths
SRC_DIR = Path.cwd().parent / 'src'
PHENO_DIR = SRC_DIR / 'phenotyping'
if str(PHENO_DIR) not in sys.path:
    sys.path.insert(0, str(PHENO_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import common evaluation utilities
from analysis.tnbc_eval_common import (
    DATA, MASK_DIR, CELLDATA_CSV, COMPOSITE_DIR,
    get_valid_pids, format_for_cp,
    compute_ap_single, extract_expression_from_mask,
    compare_expression, run_clustering, compute_gt_profiles,
    save_txt, save_excel
)

print(f'Data directory: {DATA}')
print(f'Mask directory: {MASK_DIR}')
print(f'Composite directory: {COMPOSITE_DIR}')

## Configuration

In [ ]:
# Model and phase selection
MODEL_NAME = 'microatlas'  # Options: 'microatlas', 'cellpose4', 'cellpose3', 'cellsam', 'microsam'
PHASE = 'all'              # Options: 'all', 'segment', 'express', 'cluster'
USE_GPU = True
IOU_THRESHOLD = 0.3

# MicroAtlas weights path
WEIGHTS_PATH = str(SRC_DIR.parent / 'microatlas' / 'microatlas')

out_dir = DATA / 'eval_results' / MODEL_NAME
mask_dir = out_dir / 'masks'
mask_dir.mkdir(parents=True, exist_ok=True)

# Load cell data and get valid patients
cell_df = pd.read_csv(CELLDATA_CSV)
valid_pids = get_valid_pids()
print(f'Model: {MODEL_NAME}')
print(f'Phase: {PHASE}')
print(f'Valid patients: {len(valid_pids)}')
print(f'Output: {out_dir}')

## Generate Composite Input Images

For models requiring multi-channel input, raw marker TIFs are synthesized into composites:

| Channel | Markers summed | Biological meaning |
|---|---|---|
| R | Pan-Keratin + Beta-catenin | Tumor membrane/cytoplasm |
| G | CD45 + HLA-DR | Immune membrane |
| B | dsDNA + H3K27me3 + H3K9ac | Nucleus |

In [ ]:
# Check if composite images exist, generate if needed
existing_composites = list(COMPOSITE_DIR.glob('Point*_composite.tif'))
print(f'Existing composite images: {len(existing_composites)}')

if len(existing_composites) < len(valid_pids):
    print('Generating composite images...')
    # Run the composite generation script
    import subprocess
    result = subprocess.run(
        [sys.executable, str(PHENO_DIR / 'analysis' / 'gen_composite_preview.py')],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f'Error: {result.stderr}')

## Phase 1: Segmentation & AP Evaluation

In [ ]:
if PHASE in ('all', 'segment'):
    from cellpose import models, io
    io.logger_setup()

    # Load model
    model = models.CellposeModel(gpu=USE_GPU, pretrained_model=WEIGHTS_PATH)

    ap_records = []
    for pid in valid_pids:
        t_start = time.time()

        # Load composite image
        composite = tifffile.imread(str(COMPOSITE_DIR / f'Point{pid:02d}_composite.tif'))
        img_cp = format_for_cp(composite)

        # Load GT mask
        gt_path = MASK_DIR / f'p{pid}_labeledcellData.tiff'
        gt_mask = tifffile.imread(str(gt_path)).astype(np.uint16)

        # Run segmentation
        masks_pred = model.eval([img_cp], diameter=30., channels=None, niter=1000,
                                batch_size=64, bsize=256)[0]
        pred_mask = masks_pred[0].astype(np.uint16)

        # Save prediction
        out_path = mask_dir / f'p{pid}_pred.tif'
        tifffile.imwrite(str(out_path), pred_mask, compression='deflate')

        # Compute AP
        ap05, ap075, ap09 = compute_ap_single(gt_mask, pred_mask)
        elapsed = time.time() - t_start
        print(f'  Point {pid:2d}: AP@0.5={ap05:.4f}  AP@0.75={ap075:.4f}  AP@0.9={ap09:.4f}  ({elapsed:.0f}s)')
        ap_records.append({'Patient': pid, 'AP@0.5': ap05, 'AP@0.75': ap075, 'AP@0.9': ap09})

    # Summary
    df_ap = pd.DataFrame(ap_records)
    mean_ap = df_ap[['AP@0.5', 'AP@0.75', 'AP@0.9']].mean()
    print(f'\nOverall Mean: AP@0.5={mean_ap["AP@0.5"]:.4f}  AP@0.75={mean_ap["AP@0.75"]:.4f}  AP@0.9={mean_ap["AP@0.9"]:.4f}')

    # Save results
    save_excel(out_dir / 'ap_per_image.xlsx', {'AP_Scores': df_ap})
else:
    print('Skipping segmentation phase')

## Phase 2: Expression Extraction & Comparison

Extract per-cell mean intensity from predicted masks, compare with GT expression via IoU matching.

In [ ]:
if PHASE in ('all', 'express'):
    expr_dir = out_dir / 'pred_expr'
    expr_dir.mkdir(parents=True, exist_ok=True)
    all_pred_expr = {}

    all_comp_records = []
    for pid in valid_pids:
        gt_path = MASK_DIR / f'p{pid}_labeledcellData.tiff'
        pred_path = mask_dir / f'p{pid}_pred.tif'
        if not pred_path.exists():
            continue

        # Extract expression from predicted mask
        df_gt, n_gt = extract_expression_from_mask(gt_path, pid)
        df_pred, n_pred = extract_expression_from_mask(pred_path, pid)
        all_pred_expr[pid] = df_pred

        # Save predicted expression
        df_pred.to_parquet(expr_dir / f'p{pid}.parquet')

        # Compare with GT via IoU matching
        comp, matched_df = compare_expression(gt_path, pred_path, pid, cell_df, iou_thresh=IOU_THRESHOLD)
        matched_df.to_parquet(expr_dir / f'p{pid}_matched.parquet')
        n_matched = len(matched_df)

        print(f'  Point {pid:2d}: GT={n_gt}, Pred={n_pred}, Matched(IoU>{IOU_THRESHOLD})={n_matched}')
        all_comp_records.append({
            'Patient': pid, 'n_gt_cells': n_gt, 'n_pred_cells': n_pred, 'n_matched': n_matched,
        })

    if all_comp_records:
        df_comp = pd.DataFrame(all_comp_records)
        save_excel(out_dir / 'expression_comparison.xlsx', {'Cell_Counts': df_comp})
        print(f'\nTotal matched cells: {df_comp["n_matched"].sum():,}')
else:
    # Load existing expression data for clustering phase
    expr_dir = out_dir / 'pred_expr'
    all_pred_expr = {}
    for pid in valid_pids:
        pf = expr_dir / f'p{pid}.parquet'
        if pf.exists():
            all_pred_expr[pid] = pd.read_parquet(pf)
    print(f'Loaded expression data for {len(all_pred_expr)} patients')
    print('Skipping expression phase')

## Phase 3: FlowSOM Hierarchical Clustering

Three-level immune phenotyping:
- Level 1: Immune vs Non-immune
- Level 2: Non-immune subtypes (6 classes)
- Level 3: Immune subtypes (12 classes)

Evaluated via Hungarian-matched Accuracy and Macro F1.

In [ ]:
if PHASE in ('all', 'cluster'):
    print('Running FlowSOM hierarchical clustering...')

    # Compute GT profiles
    gt_profiles, gt_counts = compute_gt_profiles(cell_df)

    # Concatenate all predicted expressions
    all_expr = []
    for pid in valid_pids:
        if pid in all_pred_expr:
            all_expr.append(all_pred_expr[pid])
        else:
            pf = expr_dir / f'p{pid}.parquet'
            if pf.exists():
                all_expr.append(pd.read_parquet(pf))

    if all_expr:
        expr_concat = pd.concat(all_expr, ignore_index=True)

        # Run clustering
        result_df, per_patient_df = run_clustering(
            expr_concat, cell_df, gt_profiles, gt_counts,
            MODEL_NAME, out_dir, iou_thresh=IOU_THRESHOLD
        )

        # Display results
        print(f'\n{"Level":<10s} {"N_GT":>6s} {"N_Matched":>10s} {"N_Valid":>8s} {"N_Correct":>10s} {"Accuracy":>9s} {"Macro_F1":>9s}')
        print('-' * 65)
        for _, r in result_df.iterrows():
            print(f'{str(r["level"]):<10s} {int(r["n_gt"]):>6d} {int(r["n_matched"]):>10d} '
                  f'{int(r["n_valid"]):>8d} {int(r["n_correct"]):>10d} '
                  f'{r["accuracy"]:>9.4f} {r["macro_f1"]:>9.4f}')

        # Save results
        save_excel(out_dir / 'clustering_results.xlsx',
                   {'Clustering': result_df, 'PerPatient': per_patient_df})
    else:
        print('No expression data available for clustering')
else:
    print('Skipping clustering phase')

## Results Summary

In [ ]:
print(f'\nResults saved to: {out_dir}')
print(f'\nOutput files:')
for f in sorted(out_dir.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        if size > 1024*1024:
            print(f'  {f.relative_to(out_dir)}: {size/1024/1024:.1f} MB')
        else:
            print(f'  {f.relative_to(out_dir)}: {size/1024:.1f} KB')